**Phase 3b:  Upload to Cosmos Gremlin**
- Upload to Cosmos Gremlin

**Enviroment variables setup**

In [ ]:
import os

os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<YOUR_OPENAI_RESOURCE>.services.ai.azure.com/"
os.environ["AZURE_OPENAI_API_KEY"] = "<YOUR_OPENAI_KEY>"
os.environ["AZURE_OPENAI_DEPLOYMENT"] = "gpt-4o"
os.environ["AZURE_OPENAI_API_VERSION"] = "2025-01-01-preview"
os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"] = "text-embedding-3-small"
os.environ["ADLS_STORAGE_ACCOUNT"] = "<YOUR_ADLS_ACCOUNT>"
os.environ["ADLS_CONTAINER"] = "<YOUR_ADLS_CONTAINER>"
os.environ["ADLS_STORAGE_KEY"] = "<YOUR_ADLS_KEY>"
os.environ["COSMOS_GREMLIN_ENDPOINT"] = "wss://<YOUR_COSMOS_ACCOUNT>.gremlin.cosmos.azure.com:443/"
os.environ["COSMOS_GREMLIN_KEY"] = "<YOUR_COSMOS_KEY>"
os.environ["COSMOS_DATABASE"] = "email_cosmos_db"
os.environ["COSMOS_GRAPH"] = "email_cosmos_graph"
os.environ["COSMOS_NOSQL_ENDPOINT"] = ""
os.environ["COSMOS_NOSQL_KEY"] = ""
os.environ["AZURE_SEARCH_ENDPOINT"] = "https://<YOUR_SEARCH_SERVICE>.search.windows.net"
os.environ["AZURE_SEARCH_API_KEY"] = "<YOUR_SEARCH_KEY>"
os.environ["AZURE_SEARCH_INDEX_NAME"] = "kg-chunks"
os.environ["AZURE_SEARCH_ENTITY_INDEX_NAME"] = "kg-entities"
os.environ["AZURE_SEARCH_COMMUNITY_INDEX_NAME"] = "kg-communities"
os.environ["PIPELINE_MODE"] = "llm"
os.environ["PROCESS_ATTACHMENTS"] = "true"
print("Environment variables set.")

**Download code from ADLS**

In [ ]:
import os
import sys
from azure.identity import DefaultAzureCredential
from azure.storage.filedatalake import DataLakeServiceClient

service = DataLakeServiceClient(
    account_url=f"https://{os.environ['ADLS_STORAGE_ACCOUNT']}.dfs.core.windows.net",
    credential=os.environ["ADLS_STORAGE_KEY"]
)

fs = service.get_file_system_client(os.environ["ADLS_CONTAINER"])

for prefix in ["code/src", "code/config"]:
    for p in fs.get_paths(path=prefix, recursive=True):
        if not p.is_directory:
            local_path = p.name.replace("code/", "", 1)
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            with open(local_path, "wb") as f:
                file_client = fs.get_file_client(p.name)
                f.write(file_client.download_file().readall())

sys.path.insert(0, ".")
src_count = len(os.listdir("src")) if os.path.exists("src") else 0
config_count = len(os.listdir("config")) if os.path.exists("config") else 0
print(f"Downloaded: {src_count} items in src/, {config_count} items in config/")

**ADLS Adapter Init**

In [ ]:
import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

sys.path.insert(0, ".")

from src.storage import ADLSAdapter

adapter = ADLSAdapter()

**Download knowledge_graph from ADLS**

In [ ]:
import os

GOLD_PATH = "gold_llm"
LOCAL_GOLD = "/tmp/pipeline/gold"

os.makedirs(LOCAL_GOLD, exist_ok=True)

# Download knowledge_graph files
kg_files = adapter.list_files(f"{GOLD_PATH}/knowledge_graph", "*")
for f in kg_files:
    rel_path = f.replace(f"{GOLD_PATH}/", "", 1)
    local_file = os.path.join(LOCAL_GOLD, rel_path)
    os.makedirs(os.path.dirname(local_file), exist_ok=True)
    with open(local_file, "wb") as fh:
        fh.write(adapter.read_bytes(f))

print(f"Downloaded {len(kg_files)} knowledge_graph files from ADLS")

**Clean Existing Test Runs Data**

**Cell 1: Clear Cosmos Gremlin**
wrote this cell to clean the test data and some junk node/entity names which llm produced with / and // for example file paths as entities. this could be hnalded in the prompt in future, but i do not want to re run the silver for this as it is slow and costly and only few nodes/entites. 


In [ ]:
from src.storage.cosmos_adapter import CosmosAdapter
import nest_asyncio
nest_asyncio.apply()

cosmos = CosmosAdapter.from_env()


**Clean if needed**

**Cell 2: Clear AI Search indexes**

**Upload to Cosmos Gremlin**

In [ ]:


# Load nodes and edges from Gold
with open(f"{LOCAL_GOLD}/knowledge_graph/nodes.json") as f:
    nodes_data = json.load(f)
with open(f"{LOCAL_GOLD}/knowledge_graph/edges.json") as f:
    edges_raw = json.load(f)

edges = list(edges_raw.values()) if isinstance(edges_raw, dict) else edges_raw

print(f"Nodes: {len(nodes_data)}, Edges: {len(edges)}")


# Upload nodes
nodes_list = []
for node_id, node in nodes_data.items():
    nodes_list.append({
        "node_id": node_id,
        "name": node.get("name", ""),
        "node_type": node.get("node_type", "UNKNOWN"),
        "properties": {
            "mention_count": node.get("mention_count", 0),
        },
    })

node_count = cosmos.bulk_upsert_nodes(nodes_list,direct_insert=True)
print(f"Uploaded {node_count} nodes to Gremlin")

# Upload edges
edges_list = []
for edge in edges:
    edges_list.append({
        "source_id": edge.get("source_id", ""),
        "target_id": edge.get("target_id", ""),
        "edge_type": edge.get("edge_type", "RELATED_TO"),
        "properties": {
            "weight": edge.get("weight", 1.0),
        },
    })

edge_count = cosmos.bulk_upsert_edges(edges_list, direct_insert=True)
print(f"Uploaded {edge_count} edges to Gremlin")